# Un bastón, tres compartimentos: de la corriente al potencial

**Objetivo:** construir el balance O-I-T, estudiar una fotocorriente impuesta y
entender qué cambia al modificar capacitancia, fuga y acoplamiento.

Al terminar podrás explicar por qué una variación de corriente negativa
hiperpolariza O y cómo se transmite a I y T. Trabajamos con un bastón **pasivo**:
la terminal es eléctrica, sin liberación sináptica ni potenciales de acción.

**Contrato científico:** A y el tiempo al máximo se toman como referencia de
bastones de sapo de Baylor; C, g y E son ilustrativos y no calibrados.
Todas las curvas de este notebook son simulaciones o funciones impuestas.
La ruta futura bastón → cono → amacrina no se implementa en esta etapa.

## 1. Preparar y recorrer el experimento

Instala desde la raíz con `python -m pip install -e '.[dev,widgets]'` y selecciona
el kernel del entorno `.venv`. Ejecuta todas las celdas en orden. Los widgets son
opcionales: las mismas comparaciones funcionan cambiando argumentos en código.

El notebook importa el modelo de `src/neurofisiologia/`; allí reside la única
implementación del sistema. No necesita los documentos privados de `Bibliografia/`.

In [ ]:
%matplotlib inline
from dataclasses import replace
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from neurofisiologia import (
    RodParameters, PhotonEvents, photon_kernel, poisson_events,
    dark_equilibrium, current_balance, simulate,
)
from neurofisiologia.examples import example_stimuli
from neurofisiologia.plotting import plot_simulation
from neurofisiologia.stimulus import BAYLOR_CELL1_AMPLITUDE_PA, BAYLOR_CELL1_T_PEAK_MS

# Funciona al abrir desde la raíz o desde notebooks/.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "src/neurofisiologia").is_dir()), None)
if root is None:
    raise RuntimeError("Abre este notebook dentro del repositorio instalado.")
output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)

## 2. Dibujar los caminos de la corriente

`Luz → corriente impuesta en O ↔ I ↔ T`

O es el segmento externo; I, el interno; T, la terminal. Cada nodo almacena
carga en una capacitancia $C_j$ y tiene una fuga $g_{L,j}$ hacia $E_j$.
Solo existen las conexiones O-I e I-T. No hay conexión directa O-T.

Usamos **corriente entrante positiva**. Una corriente axial entra a O cuando
$V_I>V_O$: $I_{I\to O}=g_{OI}(V_I-V_O)$. La misma corriente sale de I.
La fuga entrante es $-g_{L,j}(V_j-E_j)$: se opone a separarse de $E_j$.

Para capacitancia constante, $Q_j=C_j V_j$ y conservación de carga da
$dQ_j/dt=C_j dV_j/dt=\sum I_{\mathrm{entrantes},j}$. Por tanto:

$$C_O\frac{dV_O}{dt}=-g_{L,O}(V_O-E_O)+g_{OI}(V_I-V_O)+I_{photo}(t)$$
$$C_I\frac{dV_I}{dt}=-g_{L,I}(V_I-E_I)+g_{OI}(V_O-V_I)+g_{IT}(V_T-V_I)$$
$$C_T\frac{dV_T}{dt}=-g_{L,T}(V_T-E_T)+g_{IT}(V_I-V_T).$$

Al sumar los tres balances, los términos axiales se cancelan. El acoplamiento
redistribuye carga; las fugas y la entrada impuesta intercambian corriente con
el exterior. Dividir cada balance por su propia C da la EDO.

**Antes de calcular:** si O se vuelve más negativo que I, ¿qué signo tiene la
corriente axial que entra a I? Escríbelo usando $g_{OI}(V_O-V_I)$.

## 3. Unidades, procedencia y parámetros editables

$$\mathrm{pF\,mV/ms=pA},\qquad \mathrm{nS\,mV=pA}.$$

| Parámetro | Valor inicial | Procedencia |
| --- | --- | --- |
| A | 1.2 pA | Baylor, célula 1, tabla 1 p. 621 / fig. 3 p. 618 |
| $t_{peak}$ | 2200 ms = 2.2 s | Misma célula, fig. 4 p. 619 |
| C (O, I, T) | (20, 15, 5) pF | Ilustrativo, no calibrado |
| Fuga (O, I, T) | (0.05, 0.10, 0.05) nS | Ilustrativo, no calibrado |
| E (O, I, T) | (-40, -40, -40) mV | Oscuridad idealizada, no calibrada |
| $g_{OI},g_{IT}$ | 0.10, 0.10 nS | Ilustrativos, no calibrados |

Los valores eléctricos son totales por compartimento. Cambia la siguiente
celda para estudiarlos. Con E diferentes, el equilibrio oscuro se calcula
resolviendo el balance acoplado; no es simplemente la lista de E.
La corriente oscura de fondo está representada de forma idealizada por el reposo:
$I_{photo}$ es únicamente la **variación** debida a luz.

Consulta [referencias y parámetros](../docs/referencias_y_parametros.md) para
las fuentes públicas y la precisión sobre el promedio $\sigma_1=0.20$ pA de
eventos dentro de las células; no es una incertidumbre de C, g o E.

In [ ]:
# Parámetros eléctricos ILUSTRATIVOS Y NO CALIBRADOS.
p = RodParameters(
    capacitance_pf=(20.0, 15.0, 5.0),
    leak_ns=(0.05, 0.10, 0.05),
    reversal_mv=(-40.0, -40.0, -40.0),
    g_oi_ns=0.10,
    g_it_ns=0.10,
)
amplitude_pa = BAYLOR_CELL1_AMPLITUDE_PA
t_peak_ms = BAYLOR_CELL1_T_PEAK_MS
time_ms = np.linspace(0, 16000, 801)
rest_mv = dark_equilibrium(p)
print("Reposo calculado (O, I, T), mV:", rest_mv)
print("C/fuga (sin acoplamiento), ms:", np.array(p.capacitance_pf)/p.leak_ns)

## 4. Separar fotocorriente y voltaje

$$I_{photo}(t)=-\sum_k A_k h(t-t_k),\quad
h(u)=\begin{cases}(u/t_{peak})^3\exp[3(1-u/t_{peak})],&u\ge0,\\0,&u<0.\end{cases}$$

$h$ es causal, vale cero al inicio y tiene máximo unitario en $u=t_{peak}$:
para $u>0$, $d\log h/du=3/u-3/t_{peak}$. El factor $u^3$ corresponde a una
aproximación fenomenológica de cuatro etapas, con escala temporal
$\tau=t_{peak}/3$. No identifica cuatro reacciones bioquímicas reales.

Un evento efectivo impone una variación negativa cuya amplitud máxima es A.
Los potenciales se obtienen **después** resolviendo las EDO: su máximo de
hiperpolarización puede retrasarse respecto del máximo de corriente.
La figura siguiente muestra una función aproximada, no una traza medida.

In [ ]:
u_ms = np.linspace(-1000, 10000, 1101)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(u_ms, photon_kernel(u_ms, t_peak_ms), label="h impuesta, máximo unitario")
ax.axvline(t_peak_ms, color="gray", linestyle="--", label=f"Escala del experimento: {t_peak_ms:g} ms")
ax.set(xlabel="Tiempo desde el evento (ms)", ylabel="h (adimensional)",
       title="Función fenomenológica de cuatro etapas | Sin ajuste de una traza")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 5. Comprobar la oscuridad antes de aplicar luz

Sin eventos, la variación fotosensible es cero. Al iniciar en el equilibrio,
los tres voltajes deben permanecer constantes. Esta comprobación también
detecta errores de signo o inicialización. La integración usa `solve_ivp`
sobre la desviación respecto del reposo para controlar el error en respuestas
pequeñas; la salida sigue siendo el potencial absoluto en mV.

In [ ]:
dark = simulate(time_ms, p)
np.testing.assert_allclose(dark.voltage_mv, np.repeat(rest_mv[:, None], len(time_ms), axis=1), atol=1e-8)
np.testing.assert_allclose(current_balance(0, rest_mv, p, PhotonEvents()), 0, atol=1e-12)
print("Oscuridad: balance y potenciales constantes verificados.")

## 6. Resolver los cuatro escenarios reproducibles

1. Oscuridad.
2. Un evento efectivo en 1000 ms.
3. Tres eventos efectivos en 1000, 4000 y 6500 ms.
4. Iluminación tenue: eventos Poisson a 0.3 eventos/s en [1000, 11000) ms,
   con semilla 2026.

Se sortea un número Poisson de eventos y, condicionado a ese número, tiempos
uniformes ordenados. Esta tasa es de **eventos efectivos**, sin conversión desde
flujo de fotones incidentes. Una ventana Poisson puede quedar vacía.
Los eventos no son spikes neuronales. Se observan colas de corriente después
del final de la iluminación; el núcleo no se corta artificialmente.

In [ ]:
stimuli = example_stimuli()
# Las referencias de A y t_peak también se pueden modificar en la celda 3.
stimuli = {name: PhotonEvents.from_times(s.times_ms, amplitude_pa, t_peak_ms)
           for name, s in stimuli.items()}
results = {}
for name, stimulus in stimuli.items():
    result = simulate(time_ms, p, stimulus)
    results[name] = result
    fig, axes = plot_simulation(result, title=name.replace("_", " "))
    fig.savefig(output_dir / f"notebook_{name}.png", dpi=160)
    plt.show()
    print(name, "| eventos (ms):", np.round(stimulus.times_ms, 1))

## 7. Leer los resultados

La corriente y el voltaje tienen unidades distintas: una variación de 1.2 pA
no implica una variación de 1.2 mV. C regula la rapidez y g limita y transmite
la respuesta. Las curvas de I y T responden por acoplamiento, sin entrada directa.

**Predice antes de comprobar:** al poner ambos acoplamientos a cero, ¿I y T
seguirán respondiendo a luz aplicada en O? ¿Qué pasará con la amplitud de O?
Para E iguales, la primera respuesta debe ser hiperpolarizante respecto del reposo.
Si cambias E, compara siempre con `dark_equilibrium(p)`.

In [ ]:
single = results["foton_individual"]
delta_single = single.voltage_mv - rest_mv[:, None]
assert delta_single[0].min() < 0
assert np.max(delta_single) < 1e-6
print("Evento individual: mínimo ΔV (O, I, T), mV:", np.round(delta_single.min(axis=1), 4))
print("Instante del mínimo (O, I, T), ms:", time_ms[np.argmin(single.voltage_mv, axis=1)])

isolated_p = replace(p, g_oi_ns=0, g_it_ns=0)
isolated = simulate(time_ms, isolated_p, stimuli["foton_individual"])
isolated_rest = dark_equilibrium(isolated_p)
np.testing.assert_allclose(isolated.voltage_mv[1:],
                          np.repeat(isolated_rest[1:, None], len(time_ms), axis=1), atol=1e-8)
fig, axes = plot_simulation(isolated, title="Sin acoplamiento O-I ni I-T")
plt.show()

## 8. Comparar intensidad efectiva y acoplamiento

La función siguiente conserva la semilla y cambia tres controles:

- **Factor de amplitud**: modifica A como experimento numérico; 1 conserva la
  referencia de la célula 1. No significa que un fotón real tenga más energía
  de respuesta por subir el control.
- **Factor de acoplamiento**: multiplica ambos g axiales. C y fuga se conservan.
- **Tasa de eventos**: cambia la frecuencia efectiva Poisson en eventos/s;
  no incorpora saturación ni adaptación.

La línea continua es el escenario modificado; la discontinua conserva los
parámetros eléctricos originales, para el **mismo tren de eventos**. Cambia una
variable cada vez y explica las diferencias antes de cambiar la siguiente.

In [ ]:
def compare_parameters(amplitude_scale=1.0, coupling_scale=1.0, rate_hz=0.3):
    stimulus = poisson_events(rate_hz, 10000, seed=2026, start_ms=1000,
                              amplitude_pa=amplitude_pa*amplitude_scale, t_peak_ms=t_peak_ms)
    modified_p = replace(p, g_oi_ns=p.g_oi_ns*coupling_scale,
                         g_it_ns=p.g_it_ns*coupling_scale)
    baseline = simulate(time_ms, p, stimulus)
    modified = simulate(time_ms, modified_p, stimulus)
    fig, axes = plot_simulation(modified, title=f"A × {amplitude_scale:g}; g × {coupling_scale:g}; {rate_hz:g} eventos/s")
    for voltage, line in zip(baseline.voltage_mv, axes[2].lines[:3]):
        axes[2].plot(time_ms, voltage, linestyle="--", color=line.get_color(),
                     alpha=0.65, label="Original " + line.get_label()[0])
    axes[2].legend(ncol=2)
    plt.show()
    print(f"{len(stimulus.times_ms)} eventos; amplitud impuesta por evento = {amplitude_pa*amplitude_scale:g} pA")
    print("Mínimo potencial modificado (O, I, T), mV:", np.round(modified.voltage_mv.min(axis=1), 4))
    return modified

# Ejemplo sin depender de widgets: acoplamiento duplicado.
comparison = compare_parameters(coupling_scale=2)

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("Widgets opcionales ausentes: ejecuta compare_parameters con otros argumentos.")
else:
    controls = widgets.interactive(
        compare_parameters,
        {"manual": True, "manual_name": "Simular comparación"},
        amplitude_scale=widgets.FloatSlider(value=1, min=0.2, max=2, step=0.2,
                                            description="Factor A", continuous_update=False),
        coupling_scale=widgets.FloatSlider(value=1, min=0, max=3, step=0.25,
                                           description="Factor g", continuous_update=False),
        rate_hz=widgets.FloatSlider(value=0.3, min=0, max=0.6, step=0.1,
                                    description="Eventos/s", continuous_update=False),
    )
    display(controls)

## 9. Estudiar C y la resolución numérica

Una C mayor suele filtrar y retrasar la respuesta, pero en un sistema acoplado
no basta una sola constante C/g. Duplica C de O, resuelve de nuevo y compara
la amplitud y el instante del mínimo con el escenario original. Todo cambio
eléctrico continúa siendo ilustrativo.

La malla `time_ms` indica dónde guardar salida. `solve_ivp` usa pasos adaptativos
internos; `max_step_ms` solo puede reducir el límite `t_peak/30`. Comparar con
un paso máximo menor permite comprobar convergencia sin cambiar el modelo.

In [ ]:
larger_c = replace(p, capacitance_pf=(2*p.capacitance_pf[0], *p.capacitance_pf[1:]))
slow = simulate(time_ms, larger_c, stimuli["foton_individual"])
for label, result in [("C original", single), ("C_O duplicada", slow)]:
    index = np.argmin(result.voltage_mv[0])
    print(label, "| mínimo V_O (mV):", round(result.voltage_mv[0, index], 4),
          "| instante (ms):", result.time_ms[index])

refined = simulate(time_ms, p, stimuli["foton_individual"], max_step_ms=5)
error_mv = np.max(np.abs(refined.voltage_mv-single.voltage_mv))
print("Diferencia máxima con paso interno <=5 ms (mV):", error_mv)
assert error_mv < 1e-5

## 10. Ejercicios para modificar el modelo mínimo

1. Mantén E iguales; pon $g_{IT}=0$ y comprueba que T permanece en reposo.
2. Cambia solo la fuga de O. Compara amplitud y recuperación con C fija.
3. Usa E diferentes y verifica el balance oscuro antes de aplicar luz.
4. Superpone dos eventos en el mismo instante: comprueba que la desviación de
   voltaje se duplica. Explica por qué esa linealidad deja de ser fisiológica
   a intensidades altas.
5. Repite el Poisson con la misma semilla y luego con otra. Distingue
   reproducibilidad del cálculo de variabilidad entre realizaciones.

## Límites de interpretación y fuentes

Las figuras no predicen voltajes reales calibrados, eficiencia cuántica ni
liberación de glutamato. No contienen datos digitalizados. La referencia de
corriente no determina las capacitancias y conductancias. Se omiten canales
activos, ruido oscuro, bioquímica, saturación y adaptación.

Fuentes: [Baylor, Lamb y Yau (1979)](https://doi.org/10.1113/jphysiol.1979.sp012716),
[Herrera-Valdez (2026), v1](https://doi.org/10.64898/2026.08.21.746364),
[Hodgkin y Huxley (1952)](https://doi.org/10.1113/jphysiol.1952.sp004764).
El balance multicompartmental de Herrera-Valdez v1 está en la ecuación 38,
p. 18; la 36 citada en el Issue trata la curvatura. Las EDO aquí conservan
exactamente el balance del Issue #1.

Para reproducir las verificaciones: `python -m pytest -q` y
`python scripts/execute_notebook.py --kernel neurofisiologia`. Las figuras y
la copia ejecutada se guardan en `outputs/`, fuera de Git.